# Sprint 1 - Airline Passenger Satisfaction

Notebook realizuje: ingest CSV do SQLite, EDA, podzial train/val/test oraz baseline model klasyfikacyjny.

In [ ]:
from pathlib import Path
import json
import sqlite3

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

In [ ]:
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
raw_dir = project_root / 'data' / '01_raw'
dataset_csv_path = raw_dir / 'airline_passenger_satisfaction.csv'
db_path = raw_dir / 'airline_passenger_satisfaction.sqlite'
metrics_path = project_root / 'reports' / 'metrics' / 'baseline_metrics.json'

raw_dir.mkdir(parents=True, exist_ok=True)
metrics_path.parent.mkdir(parents=True, exist_ok=True)

if not dataset_csv_path.exists():
    raise FileNotFoundError(
        f'Brak pliku CSV. Umiesc airline_passenger_satisfaction.csv w: {raw_dir}'
    )

print(f'Project root: {project_root}')
print(f'Dataset CSV: {dataset_csv_path}')
print(f'SQLite DB: {db_path}')

In [ ]:
raw_df = pd.read_csv(dataset_csv_path)

with sqlite3.connect(db_path) as conn:
    raw_df.to_sql('passengers', conn, if_exists='replace', index=False)

with sqlite3.connect(db_path) as conn:
    train_df = pd.read_sql_query('SELECT * FROM passengers', conn)

print('SQLite ingest complete.')
print(f'dataset shape: {train_df.shape}')

In [ ]:
train_df.head()

In [ ]:
train_df.describe(include='all').transpose()

In [ ]:
missing_counts = train_df.isna().sum().sort_values(ascending=False)
duplicate_count = int(train_df.duplicated().sum())

print('Top missing values:')
display(missing_counts.head(15))
print(f'Duplicate rows in train: {duplicate_count}')

In [ ]:
target_col = 'satisfaction'
if target_col not in train_df.columns:
    raise ValueError(f'Kolumna target {target_col} nie istnieje w danych train.')

id_candidates = ['Unnamed: 0', 'id', 'ID']
feature_df = train_df.drop(columns=[c for c in id_candidates if c in train_df.columns], errors='ignore').copy()

## Wizualizacje EDA

In [ ]:
# 1) Rozklad klasy target
plt.figure(figsize=(7, 4))
sns.countplot(data=feature_df, x=target_col)
plt.title('Distribution of satisfaction classes')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# 2) Histogram wybranej cechy numerycznej
numeric_cols = feature_df.select_dtypes(include=['number']).columns.tolist()
hist_col = 'Age' if 'Age' in numeric_cols else (numeric_cols[0] if numeric_cols else None)
if hist_col is None:
    raise ValueError('Brak cech numerycznych do histogramu.')

plt.figure(figsize=(7, 4))
sns.histplot(data=feature_df, x=hist_col, bins=30, kde=True)
plt.title(f'{hist_col} distribution')
plt.tight_layout()
plt.show()

In [ ]:
# 3) Boxplot wybranej cechy numerycznej wzgledem targetu
box_col = 'Departure Delay in Minutes' if 'Departure Delay in Minutes' in numeric_cols else hist_col

plt.figure(figsize=(8, 4))
sns.boxplot(data=feature_df, x=target_col, y=box_col)
plt.title(f'{box_col} by {target_col}')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# 4) Rozklad wybranej cechy kategorycznej wzgledem targetu
categorical_cols = [c for c in feature_df.select_dtypes(exclude=['number']).columns if c != target_col]
cat_col = 'Customer Type' if 'Customer Type' in categorical_cols else (categorical_cols[0] if categorical_cols else None)

if cat_col is None:
    plot_df = feature_df.assign(target_for_visual=feature_df[target_col])
    cat_col = 'target_for_visual'
else:
    plot_df = feature_df

plt.figure(figsize=(7, 4))
sns.countplot(data=plot_df, x=cat_col, hue=target_col)
plt.title(f'{cat_col} by {target_col}')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# 5) Korelacje cech numerycznych
corr_cols = numeric_cols[:]
if len(corr_cols) == 0:
    feature_df = feature_df.copy()
    feature_df['constant_col'] = 1
    corr_cols = ['constant_col']
elif len(corr_cols) == 1:
    feature_df = feature_df.copy()
    feature_df['constant_col'] = 1
    corr_cols = [corr_cols[0], 'constant_col']

corr = feature_df[corr_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation heatmap (numeric features)')
plt.tight_layout()
plt.show()

## Split 70/15/15 oraz baseline model

In [ ]:
X = feature_df.drop(columns=[target_col])
y = feature_df[target_col]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print('Split sizes:')
print(f'train: {X_train.shape[0]} ({X_train.shape[0]/len(X):.2%})')
print(f'val:   {X_val.shape[0]} ({X_val.shape[0]/len(X):.2%})')
print(f'test:  {X_test.shape[0]} ({X_test.shape[0]/len(X):.2%})')

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)),
])

baseline_model.fit(X_train, y_train)

In [ ]:
y_val_pred = baseline_model.predict(X_val)

metrics = {
    'accuracy': float(accuracy_score(y_val, y_val_pred)),
    'precision_macro': float(precision_score(y_val, y_val_pred, average='macro', zero_division=0)),
    'recall_macro': float(recall_score(y_val, y_val_pred, average='macro', zero_division=0)),
    'f1_macro': float(f1_score(y_val, y_val_pred, average='macro', zero_division=0)),
}

unique_classes = sorted(pd.Series(y_train).dropna().unique().tolist())
if len(unique_classes) == 2 and hasattr(baseline_model, 'predict_proba'):
    positive_class = unique_classes[1]
    class_to_idx = {cls: idx for idx, cls in enumerate(baseline_model.named_steps['model'].classes_)}
    if positive_class in class_to_idx:
        y_val_binary = (y_val == positive_class).astype(int)
        y_val_proba = baseline_model.predict_proba(X_val)[:, class_to_idx[positive_class]]
        metrics['roc_auc'] = float(roc_auc_score(y_val_binary, y_val_proba))
    else:
        metrics['roc_auc'] = None
else:
    metrics['roc_auc'] = None

metrics

In [ ]:
metrics_payload = {
    'dataset': 'airline_passenger_satisfaction',
    'split': {
        'train': int(X_train.shape[0]),
        'val': int(X_val.shape[0]),
        'test': int(X_test.shape[0]),
    },
    'model': 'RandomForestClassifier',
    'metrics_scope': 'validation',
    'validation_metrics': metrics,
}

with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, ensure_ascii=True, indent=2)

print(f'Metrics saved to: {metrics_path}')